# Sentiment Analysis using Gemini, Llama3, and OpenAI
## EMC Comments Analysis
### 004 Data Processing (Generative Responses)

Read in curated comments from 1 to N datafiles, supplied by Sonya Sachedeva, applying comment analysis from a LLM to the comments and saving results to a file for further presentation.

Presentation provided by Power BI.

### TODO
+ Are verbotten phrases being removed?
+ Added JSON_ERROR to try/catch
+ Handle prompt engineering with library, use prompt templates.
+ https://towardsdatascience.com/document-topic-extraction-with-large-language-models-llm-and-the-latent-dirichlet-allocation-e4697e4dae87

### Version History
+ v0.1 - General access, no cleansing of data, df.apply() for OpenAI API call.  Gaps in data output.
+ v0.2 - Same dataset (Sonya Sachedeva), robust cleansing, lemmatizing, and stemming.  Summary of summary for token limit solved.
+ v0.3 - Added prompt defense, PII defense, df.apply() with defensive method, dropping lemmatizing/stemming.  Added libraries such as commonregex, spacy, and transformers.
+ v0.4 - Broke into data processing versus data prepping functions.
+ v0.5 - Moved core code into "main" to support multi-processing in future, backed up original data to 'Letter Text_ORIGINAL', explored multi-processing and GPU utilization.
+ v0.6 - New dataset to process direct from CARA extract.  Sonya Sachedeva's data inputs processed with v0.5 which has been tagged.
+ v0.7 - Added embeddings, added read of coded comments and save to binary file, started analysis of Coded Comments.  Updated code for Gemini utilization.

### Data Structure

**Delimiter=”^”, the caret, for all files**

## File #1 for each year.  Core dataset.

<div style="text-align: left">

|Field                          | Data Type            | Example                                                       |
|:------------------------------|:---------------------|:--------------------------------------------------------------|
|UUID                           |String                | 'bd65600d-8669-4903-8a14-af88203add38', each UUID will be a specific comment which has cardinatlity of 1 to N for each Letter Id.  So letter Id 1234567 might have 2-3 additioal entries with different UUID's because the Text field is each independent comment.|
|ProjectId                      |Int                   | 1234                                                          |
|LetterId                       |Int                   | 1234567                                                       |
|Text                           |String                | Lotta words, cleansed by: pii, person, email, credit card, zip code, extra-characters, etc. |
|Text_Summary                   |String                | Lotta words (200), AI generated summary of Text |
|Text_LemmStop                  |String                | Lotta words, uses Text and applies Stop Words and Lemmatize.  |
|Text_LemmStop_Summary          |String                | Lotta words (200), AI generated summary of Text_LemmStop      |
|Text_Original                  |String                | Lotta words, only carriage returned removed.                  |
|Text_AIResponse                |String                | JSON response from generative tool using fully cleansed data. |
|Text_LemmStop_AIResponse       |String                | JSON response from generative tool using Lemm/Stop data.      |
|                               |                      |                                                               |

**Table 1:** Data for initial power builder interface

</div>

## File #2 for each year.  Exploring ideas related to topics and geospatial capture of sentiment (for later).  You'll be able to combine data by UUID between files.
<div style="text-align: left">
    
|Field                          | Data Type            | Example                                                       |
|:------------------------------|:---------------------|:--------------------------------------------------------------|
|UUID                           |String                | 'bd65600d-8669-4903-8a14-af88203add38', each UUID will be a specific comment which has cardinatlity of 1 to N for each Letter Id.  So letter Id 1234567 might have 2-3 additioal entries with different UUID's because the Text field is each independent comment.|
|Topics                         |String                | Comma separated list, example: bear, rights, land management  |
|Orgs                           |String                | Comma separated list, example: sierra club, nfs               |
|Category_Original              |String                | String, from CARA system, example: land management            |
|Category_Text                  |String                | String, from generative AI: example: land management          |
|Category_LemmStop              |String                | String, from generative AI: example: land management          |
|Geospatial_lat                 |String                | 32.5                                                          |
|Geospatial_lon                 |String                | -78.25                                                        |
    
**Table 2:** Supplementary Data
    
</div>

In [ ]:
# -*- coding: utf-8 -*-

### Environment Validation

Using GCP or Azure read in arrays representing minimal library requirements (which might not be present in a Google Colab environment) and install / load the libraries as required.  Additional imports for standard libraries and tailored content to follow.

In [ ]:
###########################################
#- Minimal imports to start
###########################################
try:
    import sys
    import subprocess
    import importlib.util
    import atexit
except ImportError as e:
    print("There was a problem importing the most basic libraries necessary for this code.")
    print(repr(e))
    raise SystemExit("Stop right there!")

###########################################
#- Final Exit Routine
###########################################
@atexit.register
def goodbye():
    print("GOODBYE")

###########################################
#- Cloud Environment Setup (Priming)
###########################################
# variables establishing environments
ENV_GCP=0
ENV_AZURE=1
user_input=-1
environments=["GCP", "Azure"]
    
#prompt user for environment before continuing
user_input = 1
while True:
  try:
     if user_input > -1:
         break;
     user_input = int(input("Select the environment you're running: (0) GCP (1) Azure"))     
     if user_input > 1:
         print("Not a valid choice, please try again.")
         continue;
  except ValueError:
     print("Not a valid choice, please try again.")
     continue
  else:
     print(f"Environment selected is: {environments[user_input]}")
     break 
        
############################################
#- Import a custom library, in this case a fairly useful logging framework
############################################
from pathlib import Path
debug_lib_location = Path("./")
sys.path.append(str(debug_lib_location))
import debug

libraries=["transformers", "langchain", 
           "alive-progress", "tqdm", "pyspellchecker", "wordcloud", "langchain", "icecream", "numba", 
           "fitz","dataclasses", "commonregex", "transformers", "spacy", "PyMuPDF", "PyPDF2", "pdfminer", 
           "pdfplumber","pdf2image","pytesseract"]    
debug.msg_info(f"Validating environment for the following pip packages: {libraries}")

#load environment for non-generative libraries
try:
    for library in libraries:
      if library == "Pillow":
        spec = importlib.util.find_spec("PIL")
      else:
        spec = importlib.util.find_spec(library)
      if spec is None:
        print("...installing library " + library)
        subprocess.run(["pip", "install" , library, "--quiet"])
      else:
        print("...library " + library + " already installed.")
except (subprocess.CalledProcessError, Exception) as e:
    print("Error: Failed to install required packages, your code might not run properly.")
    print(repr(e))

#load environment specific libraries for generative AI.
try:    
    if environments[user_input]=="GCP":
      subprocess.run(["pip", "install" , "--upgrade", "google-cloud-aiplatform", "--quiet"])
      gcp_libraries=["google-generativeai", "google-cloud-secret-manager"]
      for library in gcp_libraries:
        spec = importlib.util.find_spec(library)
        if spec is None:
          print("...installing library " + library)
          try:
              subprocess.run(["pip", "install" , library, "--quiet"])
          except (subprocess.CalledProcessError, Exception) as e:
             print("Error: Failed to install required packages, your code might not run properly.")
             print(repr(e))
        else:
          print("...library " + library + " already installed.")
    
        from google.cloud import aiplatform
        import vertexai.preview
        from google.cloud import secretmanager
    elif environments[user_input]=="Azure":
      azure_libraries=["openai", ]
      for library in azure_libraries:
        spec = importlib.util.find_spec(library)
        if spec is None:
          print("...installing library " + library)
          try:
              subprocess.run(["pip", "install" , library, "--quiet"])
          except (subprocess.CalledProcessError, Exception) as e:
              print("Error: Failed to install required packages, your code might not run properly.")
              print(repr(e))
        else:
          print("...library " + library + " already installed.")
    else:
        print("There was a problem processing your request.  Only numeric input of 0 or 1 is allowed.")
        print("Continued operations is not possible without the proper installed tools.")
        raise SystemExit("Stop right there!")
except Exception as e:
    print("There was a problem processing library installs for Generative AI libraries")
    print(repr(e))
    raise SystemExit("Stop right there!")

debug.msg_debug("...dynamic environment installs complete.")

## Includes and Libraries

In [ ]:
debug.msg_info("Library imports")    
############################################
# INCLUDES
############################################

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# a set of libraries that perhaps should always be in Python source
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...core libraries.")
import os
import datetime
import gc
import socket
import sys
import getopt
import inspect
import traceback
import warnings
import json
import pickle
from pathlib import Path
import itertools
import datetime
import re
import shutil
import string
from io import StringIO
import tqdm


import io
import math
import textwrap
import random
import glob
import time
from time import perf_counter
import subprocess
import backoff

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Function Profiling
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import cProfile
import pstats
import io
from pstats import SortKey

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Data Science Libraries
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...classic data science libraries.")

#optimization routines
from numba import jit
import numpy as np
import scipy as sp
#from sklearn.linear_model import LinearRegression


# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Additional libraries for this work
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...application specific libraries.")
import math
from base64 import b64decode
from IPython.display import Image
import requests
from bs4 import BeautifulSoup                 #used to parse the text
from wordcloud import WordCloud, STOPWORDS    #custom library specifically designed to make word clouds
from spellchecker import SpellChecker
import fitz
#to handle strange characters
from unidecode import unidecode 

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Graphics
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...graphics.")
#import PIL
from PIL import Image
import PIL.ImageOps
import matplotlib as matplt
import matplotlib.pyplot as plt

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# progress bar
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...progress bars.")
from alive_progress import alive_bar
#from alive_progress.styles import showtime, Show
from tqdm.notebook import trange, tqdm
#from tqdm import trange, tqdm

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
#- PII libraries (regular expressions)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...regular expressions for PII and transformers for prompt injection defense.")
from commonregex import CommonRegex
from commonregex import email
from commonregex import time
from commonregex import credit_card
from commonregex import ip
from commonregex import ipv6
from commonregex import link
from commonregex import phone
from commonregex import street_address
from commonregex import btc_address

debug.msg_debug("...spacy (pii defense).")
import spacy
from spacy.language import Language
from spacy.tokens import Doc

debug.msg_debug("...hugging face model support.")
#injection defense
from transformers import pipeline

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
#- Tensorflow AI/ML libraries (seek to use GPU's)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
#load first
try:
    debug.msg_debug("...TensorRT")    
    import tensorrt
    assert tensorrt.Builder(tensorrt.Logger())
except ImportError as ie:
    debug.msg_warning("Failed to import tensorrt, this might be a problem if trying for enhanced processing.")
    debug.msg_warning(f"...{repr(ie)}")
    pass

try:
    #load second
    debug.msg_debug("...TensorFlow")        
    import tensorflow as tf
except ImportError as ie:
    debug.msg_warning("Failed to import tensorflow, might not have a GPU or the proper environment loaded")
    debug.msg_warning(f"...{repr(ie)}")
    pass

try:
    debug.msg_debug("...CUDF")    
    import cudf
except ImportError as ie:
    debug.msg_warning("Failed to import cudf, likely don't have a GPU")
    debug.msg_warning(f"...{repr(ie)}")
    pass

try:
    debug.msg_debug("...Torch")    
    import torch
except ImportError as ie:
    debug.msg_warning("Failed to import torch, likely don't have a GPU or access to that library.")
    debug.msg_warning(f"...{repr(ie)}")
    pass


import pandas as pd
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
#- NLTK required resources
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...natural language processing.")
import nltk
from nltk.stem import PorterStemmer  # A word stemmer based on the Porter stemming algorithm.  Porter, M. "An algorithm for suffix stripping." Program 14.3 (1980): 130-137.
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.tree import tree
#from nltk.book import *
from nltk import FreqDist
from nltk import sent_tokenize, word_tokenize
from nltk.corpus import stopwords    

nltk.download('punkt')
nltk.download("words")
nltk.download("stopwords")
#nltk.download('averaged_perceptron_tagger')      #looks like you have to download select neural layers for specific functions, head to read the erorr output to learn this.


## Functions

### Numpy / Pandas Configuration Settings

In [ ]:
def set_library_configuration() -> None:
    
    ############################################
    #- JUPYTER NOTEBOOK OUTPUT CONTROL / FORMATTING
    ############################################
    #pandas set floating point to 4 places to things don't run loose
    debug.msg_info("Setting Pandas and Numpy library options.")    
    pd.set_option('display.max_colwidth', 10) # None if you want to view the full json blob in the printed dataframe, use this
    pd.options.display.float_format = '{:,.4f}'.format
    np.set_printoptions(precision=4)

## Functions

### Custom Exception Display

In [ ]:
## Manages exception output.
#  @param   (Exception)             - Exception to expound upon
#  @returns (None)                  - None
def process_exception(inc_exception) -> None:
    print(f"{BOLD_START}(Exception encountered):{BOLD_END} {type(inc_exception).__name__}")
    print(f"Details: {str(inc_exception)}")
    print("Traceback:")
    traceback.print_exc()

In [ ]:
def profile_function(func):
    def wrapper(*args, **kwargs):
        pr = cProfile.Profile()
        pr.enable()
        result = func(*args, **kwargs)
        pr.disable()
        s = io.StringIO()
        sortby = SortKey.CUMULATIVE
        ps = pstats.Stats(pr, stream=s).sort_stats(sortby)
        ps.print_stats()
        print(s.getvalue())
        return result
    return wrapper

### Library Versioning Display

In [ ]:
## Outputs library version history of effort.
#
#  @returns (None)                  - None
def lib_diagnostics() -> None:

    import pkg_resources
    
    debug.msg_info(f"Entering {__name__} {inspect.stack()[0][3]}") 
    
    package_name_length=40
    package_version_length=20

    # Get installed packages
    the_packages=["cupy", "jupyter-core", "langchain", "langchain-core", "nltk", "numba", "numpy", "pandas", "pydantic", "pyspellchecker", "spacy", "scipy", "scikit-learn", "seaborn", "usaddress", "xarray",]
    the_packages.sort()
    
    installed_dict = {pkg.key: pkg.version for pkg in pkg_resources.working_set}
    installed=list(installed_dict.keys())
    installed.sort()
    
    #for package_idx, package_name in enumerate(installed):
    for idx, name in enumerate(installed):
         if name in the_packages:
             installed_version = installed_dict[name]
             print(f"{name:<40}#: {str(pkg_resources.parse_version(installed_version)):<20}")
   
    try:
        print(f"{'TensorFlow version':<40}#: {str(tf.__version__):<20}")
        print(f"{'     gpu.count:':<40}#: {str(len(tf.config.experimental.list_physical_devices('GPU')))}")
        print(f"{'     cpu.count:':<40}#: {str(len(tf.config.experimental.list_physical_devices('CPU')))}")
    except Exception as e:
        pass

    try:
        print(f"{'Torch version':<40}#: {str(torch.__version__):<20}")
        print(f"{'     GPUs available?':<40}#: {torch.cuda.is_available()}")
        print(f"{'     count':<40}#: {torch.cuda.device_count()}")
        print(f"{'     current':<40}#: {torch.cuda.current_device()}")
    except Exception as e:
        pass


    try:
      print(f"{'OpenAI Azure Version':<40}#: {str(the_openai_version):<20}")
    except Exception as e:
      pass

    print(f"{BOLD_START}List Devices{BOLD_END} #########################################")
    try:
      from tensorflow.python.client import device_lib
      print(device_lib.list_local_devices())
      print("")
    except RuntimeError as e:
      # Visible devices must be set before GPUs have been initialized
      print(str(repr(åe)))

    print(f"{BOLD_START}Devices Counts{BOLD_END} ########################################")
    try:
      print(f"Num GPUs Available: {str(len(tf.config.experimental.list_physical_devices('GPU')))}" )
      print(f"Num CPUs Available: {str(len(tf.config.experimental.list_physical_devices('CPU')))}" )
      print("")
    except RuntimeError as e:
      # Visible devices must be set before GPUs have been initialized
      print(str(repr(e)))

    print(f"{BOLD_START}Optional Enablement{BOLD_END} ####################################")
    try:
      gpus = tf.config.experimental.list_physical_devices('GPU')
    except RuntimeError as e:
      # Visible devices must be set before GPUs have been initialized
      print(str(repr(e)))

    if gpus:
      # Restrict TensorFlow to only use the first GPU
      try:
        tf.config.experimental.set_visible_devices(gpus[0], 'GPU')
        logical_gpus = tf.config.experimental.list_logical_devices('GPU')
        print( str( str(len(gpus)) + " Physical GPUs," + str(len(logical_gpus)) + " Logical GPU") )
      except RuntimeError as e:
        # Visible devices must be set before GPUs have been initialized
        print(str(repr(e)))
      print("")
        
    debug.msg_info(f"Exiting {__name__} {inspect.stack()[0][3]}") 
    return

### Split Text

Given a set of text break the text into chunks to that is can be subprocessed by the AI due to token limit concerns.

In [ ]:
##Break the string into chunks and sub-process it
#
#  @param (incoming body of text, str)    - str      - incoming body of text
#  @param (chunk size, int)               - int      - Chunk size
#  @returns ([str])                       - [str]    - List of strings broken into the "chunks"
########################################

def split_text(text, chunk_size=8000) -> [str]:
    
  """
  Splits the given text into chunks of approximately the specified chunk size.
  
  Args:
  text (str): The text to split.
  
  chunk_size (int): The desired size of each chunk (in characters).
  
  Returns:
  List[str]: A list of chunks, each of approximately the specified chunk size.
  """
  
  chunks = []
  current_chunk = StringIO()
  current_size = 0
  sentences = sent_tokenize(text)
  for sentence in sentences:
    sentence_size = len(sentence)
    if sentence_size > chunk_size:
      while sentence_size > chunk_size:
        chunk = sentence[:chunk_size]
        chunks.append(chunk)
        sentence = sentence[chunk_size:]
        sentence_size -= chunk_size
        current_chunk = StringIO()
        current_size = 0
    if current_size + sentence_size < chunk_size:
      current_chunk.write(sentence)
      current_size += sentence_size
    else:
      chunks.append(current_chunk.getvalue())
      current_chunk = StringIO()
      current_chunk.write(sentence)
      current_size = sentence_size
  if current_chunk:
     chunks.append(current_chunk.getvalue())
  return chunks
  

### Summarize Chunked data (summary of summaries)

In [ ]:
## Main routine that executes all code, does return a data frame of data for further analysis if desired.
#
#  @param ([])      - list   - list of text chunked into a list
#  @param (int)     - int    - size of maximum tokens allowed for the model response
#  @param (str)     - str    - prompt used to ask the generative AI a question
#  @return ([])     - list   - responses from generative AI.
@backoff.on_exception(backoff.expo, Exception, max_tries=3)
def summarize(chunks, inc_max_tokens, prompt) -> [] :

  #debug.msg_debug("Entered summarize")
  summaries = []
  summary = ""
  for index,chunk in enumerate(chunks):
    completion_message=""
    resultant=""
     
    #print(f"...iteration:{index}")
    new_prompt=prompt+chunk
    #print(f"      in summarize(), length of prompt is: {len(new_prompt)}")

    ########################################
    #Model Invocation
    ########################################
    try:
        dynamic_message_text = [
                    {"role":"system", "content": "You are an experienced secretary that summarizes documents." },
                    {"role":"user",   "content": "Please summarize and extract relevant information from the following text:"  + new_prompt }
                   ]        

        
        completion_message = client.chat.completions.create(
              model=the_model,
              messages = dynamic_message_text,
              temperature=model_temperature,
              #max_tokens=model_max_tokens,
              max_tokens=inc_max_tokens+len(new_prompt),
              top_p=model_top_p,
              frequency_penalty=model_frequency_penalty,
              presence_penalty=model_presence_penalty,
              stop=None
        )
        resultant=str(completion_message.choices[0].message.content)

    except Exception as oops:
      process_exception(f"Failed to invoke summarize with chat completion api call with following error ({str(oops)})")
      resultant=f"{oops}"      

    #print(f"        Summary length is:{len(resultant)}")

    summaries.append(resultant)
    #debug.msg_debug("Exited summarize")

  return ''.join(summaries)

### Azure OpenAI API Call for Generative Answer

In [ ]:
## Azure Open AI API call for generative results
#
#  @param (str)      - str   - Incoming comment to evaluate
#  @param (int)      - int   - Maximum model tokens (model dependent)
#  @return (str)     - str   - Generative Response
@backoff.on_exception(backoff.expo, Exception, max_tries=3)
def completion_iteration_az(inc_value:str, inc_max_tokens:int) -> str:

    the_resultant=""
    try:
        # This is the structure for GPT 3.5, other models may have different syntax
        system_prompt_context="You are a feedback analyst. You parse and extract from submitted letters to help organize and structure the data."
        #user_prompt_context=user_prompt + json.dumps(EXAMPLE_JSON) + "Here is the letter text to analyze: " + inc_value
        user_prompt_context = """Analyze the following text and extract the requested information. Provide your response ONLY as a JSON object, with no additional text.
            Extract the following fields:
            
            1. Sentiment: (Required) 
               - Scale: -1.0 to 1.0
               - -1.0 is extremely negative, 0 is neutral, 1.0 is extremely positive
               - Use fractional values for nuanced sentiment (e.g., -0.3, 0.7)
            
            2. SentimentConfidence: (Required)
               - Scale: 0.0 to 1.0
               - Represents your confidence in the sentiment score
               - 0.0 is completely uncertain, 1.0 is absolutely certain
            
            3. Category: (Required)
               - Single value from the following options: """ + response_categories + """
               - Choose the most relevant category
               - If multiple categories apply, choose the most prominent one
            
            4. CitingLaw: (Required)
               - Boolean (true/false)
               - true if the text explicitly mentions or implies a law is being broken or referenced
               - false if no legal references are made
            
            5. BriefSummary: (Required)
               - 1-2 sentence summary of the key points in the letter
               - Focus on the main issue or concern expressed
            
            6. SpecificFeedback: (Required)
               - Boolean (true/false)
               - true if the submitter provides a specific suggestion or solution
               - false if the feedback is general or no solution is proposed
            
            7. ProposedResolution: (Required if SpecificFeedback is true)
               - Summarize the specific solution or action the submitter proposes
               - If no specific resolution is proposed, use "No specific resolution proposed"
            
            Guidelines:
            - Strive for consistency and accuracy in your analysis
            - If the text is ambiguous, use your best judgment and reflect this in the SentimentConfidence score
            - Ensure all required fields are filled, even if you have low confidence in some assessments
            - Base your analysis solely on the content of the provided text, not on external knowledge
            
            Your response should be a valid JSON object. Here's an example format:
            
            """ + json.dumps(EXAMPLE_JSON) +  " TEXT: " + inc_value
        
        dynamic_message_text = [
            {"role": "system", "content": f"{system_prompt_context}"},
            {"role": "user", "content": f"{user_prompt_context}"}
        ]

        #print("#######################################################################################")
        #print(f"System:{system_prompt_context}")
        #print(f"Role:{user_prompt_context}")        
        #print("#######################################################################################")
        
        completion = client.chat.completions.create(
          model=the_model,
          messages = dynamic_message_text,
          temperature=model_temperature,
          max_tokens=inc_max_tokens,
          top_p=model_top_p,
          frequency_penalty=model_frequency_penalty,
          presence_penalty=model_presence_penalty,
          stop=None
        )
        the_resultant=completion.choices[0].message.content

        #print(the_resultant)
    
    except Exception as e:
        #commenting this out for now as it's super verbose and massively scrolls the thing down
        process_exception(f"Completion_iteration_az (Azure) failed with {inc_max_tokens} atempted, exception is: {str(e)}")
        process_exception(f"...original message: {inc_value}")
        the_resultant=ERROR_PHRASE
    finally:
        the_resultant=re.sub("\n", " ", the_resultant)
        the_resultant=the_resultant.strip()
        return the_resultant


### Process Each Comment Atomically

Larger comments (beyond token max go to `process_long_comment`)

In [ ]:
## Process comment starts the comment processing mechanism called via a lambda function with a Pandas DataFrame
#
#  @param (str)  -   str     - Incoming comment
#  @return(str)  -   str     - Generative answer
def process_comment(inc_comment:str) -> str:
    the_max_tokens=model_max_tokens - len(inc_comment)+1 + len(user_prompt) + len(response_categories) + len(EXAMPLE_JSON)
    if the_max_tokens < MINIMUM_AI_RESPONSE:
       #don't calculate the actual input since it will be summarized, just everything used to augment the prompt
       the_max_tokens=int( (model_max_tokens - len(user_prompt) + len(response_categories) + len(EXAMPLE_JSON) ) /2 )
       long_resultant=process_long_input(inc_value=inc_comment, inc_max_tokens=the_max_tokens)
       the_max_tokens=model_max_tokens - len(long_resultant)+1 + len(user_prompt) + len(response_categories) + len(EXAMPLE_JSON)
       resultant=completion_iteration_az(inc_value=long_resultant, inc_max_tokens=the_max_tokens)
    else:
       resultant=completion_iteration_az(inc_value=inc_comment, inc_max_tokens=the_max_tokens)
               
    if (len(resultant) < MINIMUM_AI_RESPONSE) or (resultant.find(ERROR_PHRASE) > 0):
        subprocess.run(["sleep", f"{MINIMUM_AI_WAIT}" ])
        resultant=completion_iteration_az(inc_value=row[name], inc_max_tokens=the_max_tokens)
        
    return resultant


### Process Maxed Out Token Payload

In [ ]:
## Process input longer than normal token boundary condition, summarize and get final answer.
#
#  @param (str)    - str   - Incoming string of data that's too long
#  @param (int)    - int   - Maximum tokens allowed for processing (model specific)
#  @return (str)   - str   - Final answer from the generative summary of summaries
@backoff.on_exception(backoff.expo, Exception, max_tries=3)
def process_long_input(inc_value:str, inc_max_tokens:int) -> str:
    #debug.msg_debug("Entered process_long_input")
    summary_token_max=150
    summary_overload=0
    chunks=[]
    the_answer=ERROR_PHRASE

    # Calling the split function to split text
    #print(f"process_long_input inc_value len is:{len(inc_value)} with max tokens to work with: {inc_max_tokens}")
    chunks = split_text(inc_value, chunk_size=inc_max_tokens)
    #print(f"There are {len(chunks)} chunks.")

    #print(f"Summary overload: {len(chunks)*summary_token_max}")

    resultant=summarize(chunks, inc_max_tokens, prompt = f"Please abstratively summarize the following text in {str(summary_token_max)} words or less: \n")

    ########################################
    # Summarize the summaries
    ########################################
    try:
        the_max_tokens=model_max_tokens - len(resultant)
    
        dynamic_message_text = [
                    {"role":"system", "content": "You are feedback analyst that summarizes documents." },
                    #CGW TODO - hack to limit total payload and get the job done, need a more robust mechanism.
                    {"role":"user",   "content": "Please summarize and extract relevant information from the following text : \n" + " ".join(resultant[:the_max_tokens]) }
                   ]        

        completion_message = client.chat.completions.create(
              model=the_model,
              messages = dynamic_message_text,
              temperature=model_temperature,
              #max_tokens=model_max_tokens,
              max_tokens=inc_max_tokens,
              top_p=model_top_p,
              frequency_penalty=model_frequency_penalty,
              presence_penalty=model_presence_penalty,
              stop=None
        )
        the_answer=str(completion_message.choices[0].message.content)

    except Exception as oops:
      process_exception(f"process_long_input, exception: ({str(oops)})")
      the_answser=str(oops)

    #debug.msg_debug("Exited process_long_input")
    return ERROR_PHRASE

In [ ]:
def output_csv(inc_filename:str, inc_df: pd.DataFrame) -> None:
    
    #df=pd.DataFrame(inc_candidates) 
    output_filename=inc_filename
    debug.msg_debug(f"Saving the data to a file ({output_filename}).")
    inc_df.to_csv(output_filename, sep="^", header=True, index=False)
    

In [ ]:
def output_excel(inc_filename:str, inc_df: pd.DataFrame) -> None:

    debug.msg_info(f"Entering {__name__} {inspect.stack()[0][3]}")

    inc_df.to_excel(inc_filename, sheet_name='Sheet1', 
                    na_rep='', 
                    float_format=None, 
                    columns=None,        #sequence or list of str, optional
                    header=True,         #write out col names, bool
                    index=True, 
                    index_label=None, 
                    startrow=0, 
                    startcol=0, 
                    engine=None, 
                    merge_cells=True, 
                    inf_rep='inf', 
                    freeze_panes=None, 
                    storage_options=None, 
                    )
   
    debug.msg_debug(f"Saving the data to a file ({inc_filename}).")

    debug.msg_info(f"Exiting {__name__} {inspect.stack()[0][3]}")

In [ ]:
def setup_openai_client():
    
    debug.msg_info(f"Entering {__name__} {inspect.stack()[0][3]}")
    from openai import AzureOpenAI
    
    #model connection values for client
    debug.msg_debug("...gathering API key information.")
    try:
        the_endpoint=os.getenv("OPENAI_USFS_API_BASE")
        the_key=os.getenv("OPENAI_USFS_API_KEY")
        the_version=os.getenv("OPENAI_USFS_API_VERSION")
    except (Exception, KeyError) as e:
        process_exception(e)
        raise EnvironmentError(f"Missing environment variable: {e}")
        
    debug.msg_debug("...creating Azure client.")
    try:
        client = AzureOpenAI(
            azure_endpoint = the_endpoint,
            api_key = the_key,
            api_version=the_version,
        )
    except Exception as e:
        process_exception(e)
        raise ConnectionError(f"Failed to initialize OpenAI client: {e}")
        
    debug.msg_info(f"Entering {__name__} {inspect.stack()[0][3]}")
    return client

In [ ]:
def process():

        #import pickle
        #import sys
        #from pathlib import Path
        #import inspect
        #debug_lib_location = Path("./")
        #sys.path.append(str(debug_lib_location))
        #import debug
        
        debug.msg_info(f"Entering {__name__} {inspect.stack()[0][3]}")  
        ####################################################################################################################
        #- Input Source (comments)
        #  Read in each set of letters (hand modified) and store the results into a dataframe for further processing.
        #  Match only "unique" letters from the data domain (this is what we inherited).
        # Load the binary (saved from previous process) datafile
        ####################################################################################################################
        #establish data version, aligned with code
        data_version_release="-".join([str(VERSION_NAME), str(VERSION_MAJOR), str(VERSION_MINOR), str(VERSION_RELEASE)])
        #target_filename=f"./{data_version_release}"+"_EMC_Comments.bin"
        
        target_filename="./2014_CMTANL-0-5-0_EMC_Comments.bin"
        debug.msg_info(f"Loading data file: {target_filename}.")
        
        try:
            with open(target_filename, 'rb') as file:
                df = pickle.load(file)
                #emc_comments = pickle.load(open(target_filename, "rb"))
        except (pickle.UnpicklingError, FileNotFoundError, IOError, Exception)  as e:
            debug.msg_warning("FAILED to unpickle or find the saved binary file, you might have corruption, investigate.")
            process_exception(e)
            sys.exit(66)   #*nix file not found
        
        debug.msg_debug(f"...reloaded {target_filename}")
        #CGW FIX
        #df = df.sample(n=10) 
        
        #clean up carriage returns from the output so the final datafile is "clean" and doesn't cause prompt issues.
        df["SOURCE_COLUMN_NAME_ORIGINAL"] = df["SOURCE_COLUMN_NAME_ORIGINAL"].apply(lambda value: re.sub("\n", " ", value))
        df["SOURCE_COLUMN_NAME_ORIGINAL"] = df["SOURCE_COLUMN_NAME_ORIGINAL"].apply(lambda value: value.strip())
        
        ####################################################################################################################
        #- OpenAI Column Created, ready to receive output from Generative Calls
        ####################################################################################################################
        df[OPENAI_RESULT]=""
        
        ####################################################################################################################
        #- OpenAI Calls (Generative evaluation of comments)
        ####################################################################################################################
        
        # Apply the function to all the rows in the dataframe    
        debug.msg_debug("...started generative calls")
        target_name="ResultOPENAI"
        df[target_name] = df[SOURCE_COLUMN_NAME].apply(lambda value: process_comment(value))
        debug.msg_debug("...finished generative calls")
        
        ####################################################################################################################
        #- See what the results look like, verify the output
        ####################################################################################################################
        #debug.msg_info("Head of dataframe after processing.")
        #print(df.head())
        
        ####################################################################################################################
        #- Check for NULL OpenAI Responses
        ####################################################################################################################
        #debug.msg_debug("Empty Records based on OpenAI LLM calls")
        #debug.msg_debug(df[OPENAI_RESULT].isnull().sum())
        #debug.msg_debug(" ")
        
        
        ####################################################################################################################
        #- Save results to datafile
        ####################################################################################################################
        debug.msg_info("Saving completed data file")
        try:
            #save to textual output
            target_filename=f"./{data_version_release}"+"EMC_Comments.csv"
            output_csv(target_filename, df)
        except (pickle.UnpicklingError, FileNotFoundError, IOError, Exception)  as e:
            process_exception(f"Failed to save CSV file for completed records: {str(e)}")
        
        try:
            #save to MS Excel
            target_filename=f"./{data_version_release}"+"_EMC_Comments.xlsx"
            output_excel(target_filename, df)    
        except (pickle.UnpicklingError, FileNotFoundError, IOError, Exception)  as e:
            process_exception(f"Failed to save Excel file for completed records: {str(e)}")
        
        debug.msg_info(f"Exited {__name__} {inspect.stack()[0][3]}")

## Main

In [ ]:
#MAIN, main
if __name__ == "__main__":

    set_library_configuration()
    start_t=perf_counter()
    print("BEGIN PROGRAM")

    ############################################
    # GLOBAL CONFIGURATION
    ############################################
    #used for values outside standard ASCII, just do it, you'll need it
    ENCODING  ="utf-8"
    os.environ['PYTHONIOENCODING']=ENCODING
    #spacy requirement
    os.environ['TOKENIZERS_PARALLELISM']="false"


    debug.msg_info("Variable declaration.")    
    ############################################
    # GLOBAL VARIABLES
    ############################################
    DEBUG = 1
    DEBUG_DATA = 0
    
    # CODE CONSTRAINTS
    VERSION_NAME    = "CMTANL"
    VERSION_MAJOR   = 0
    VERSION_MINOR   = 6
    VERSION_RELEASE = 0
    
    #used for values outside standard ASCII, just do it, you'll need it
    ENCODING  ="utf-8"
    TEXT_WIDTH=77
    BOLD_START = "\033[1m"
    BOLD_END = "\033[0;0m"
    
    ###########################################
    #- API Parameters for things like WordCloud
    ###########################################
    IMG_BACKGROUND=None                        #None without quotes or "black", "white", etc...
    IMG_FONT_SIZE_MIN=14
    IMG_WIDTH=800
    IMG_HEIGHT=600
    
    ############################################
    # APPLICATION VARIABLES
    ############################################
    SPELL_CHECK_DISTANCE=2
    MINIMUM_AI_RESPONSE=25                     #words
    MINIMUM_AI_WAIT=15                         #seconds
    os.environ["MINIMUM_AI_WAIT"] = "str(MINIMUM_AI_WAIT)"
    MINIMUM_LETTER_LENGTH=15
    SOURCE_COLUMNS_NAME=["Letter Text"]        #body of text where the actual comment is
    SOURCE_COLUMNS_IDX=[ 10 ]                   #location in data frame AFTER removal of columns
    SOURCE_COLUMN_NAME=SOURCE_COLUMNS_NAME[0]
    EVALUATION_RECORDS=250
    ERROR_PHRASE = 'Error code: 400'
    OPENAI_RESULT="ResultOPENAI"
    DATA_DIR='./'                               #expect to read the binary files locally
    
    
    ############################################
    # MODEL PARAMETERS
    ############################################
    #model parameters
    the_model="gpt-35-turbo-16k"
    model_temperature=0.7
    model_max_tokens=8000
    model_top_p=0.95
    model_frequency_penalty=0
    model_presence_penalty=0
    summary_token_max=150
    
    ############################################
    # PROMPT PARAMETERS
    ############################################
    PROMPT_SUMMARY_LIMIT="200"                   #number of words to generate
    PROMPT_SUMMARY_METHOD=" abstractive "        #abstractive or extractive
    
            
    response_categories = "Air Quality, Botany, Climate Change, Cultural/Heritage, Facilities, FireFuels, Fisheries, Hydrology, Lands/Special Uses, Minerals/Geology, NEPA/Proj Development, Other/Misc, Public Engagement, Range/Weeds, Recreation, Roadless, Silviculture/Veg, SocioEconomic, Soils, Transportation, Visuals, Wilderness, Wildlife"
    user_prompt = "Extract the following information: Sentiment (required, -1 to 1, where -1 is extremely negative and 1 is extremely positive), SentimentConfidence (decimal, 0-1, represents the confidence level of your sentiment number), Category (required, single value, choices are: " + response_categories + ". CitingLaw (boolean, return True if the text implies that a law is being broken or is referencing a law), BriefSummary (required, 1-2 sentence summary of the letter), SpecificFeedback (boolean, required, true if a specific resolution to their issue is identified), ProposedResolution (summarization of what the submitter thinks will resolve their issue). Your response should be ONLY the JSON analysis, no other text. Here is an example response: "
    
    
    JSON_PAYLOAD = '{ \
      "Sentiment": 0.0 \
      "SentimentConfidence": 0.0, \
      "Category": "Some Category", \
      "CitingLaw": "TRUE", \
      "BriefSummary": "Summary information about the topic.", \
      "SpecificFeedback": "TRUE", \
      "ProposedResolution":"Proposed change for improvement." \
    }'
    
    JSON_ERROR = ' { \
        "Sentiment": 100, \
        "SentimentConfidence": 100, \
        "Category": "error", \
        "CitingLaw": "error", \
        "BriefSummary": "error", \
        "SpecificFeedback": "error", \
        "ProposedResolution": "error"\
    }'
    
    EXAMPLE_JSON = '{ \
      "Sentiment": -0.5, \
      "SentimentConfidence": 0.8, \
      "Category": "Hydrology", \
      "CitingLaw": "TRUE", \
      #"Tags": ["water quality", "pollution", "regulation"], \
      "BriefSummary": "The letter expresses concerns over river pollution impacting community health and calls for stricter regulatory oversight and sustainable practices.", \
      "SpecificFeedback": "TRUE", \
      "ProposedResolution":"Stricter regulation on industry waste dumping." \
    }'
    #you could also have it propose a response that people could review/use as a starting point here
    #"SuggestedResponse":"Thank you for your thoughtful feedback regarding the dumping regulations. We share your concern about the need for robust measures to protect our environment and public health. Please know that your comments are invaluable to us and contribute significantly to our ongoing review process aimed at strengthening our regulations. We are actively working with various stakeholders, including environmental experts and community leaders, to ensure our policies effectively address these concerns while promoting sustainable practices. To stay engaged and informed about the progress and developments in this area, we encourage you to visit our website regularly and participate in upcoming public forums. Your active involvement is essential as we strive to enhance our environmental policies for the betterment of our community and future generations."
        
    ############################################
    #- Invocation of functions and instantiation of system needs, nltk instantiation
    ############################################
    #setup the text wrapper
    debug.msg_debug(f"...Text Wrapper instantiated.")
    wrapper = textwrap.TextWrapper(width=TEXT_WIDTH)
    
    #show your libraries
    lib_diagnostics()    

    ###########################################
    #- OPENAI
    # Generative AI Library Configuration
    # Tailored for OpenAI environment on Azure for now.  API keys and other relevant information in .bashrc_keys environment variable on system for security.
    ###########################################
    client=setup_openai_client()
    
    ############################################
    #- Core routine that does the work
    ############################################
    process()

    end_t=perf_counter()
    print("END PROGRAM")
    print(f"Elapsed time: {end_t - start_t}")
